# Double Responses in Decision Tasks
Group 17

In [1]:
import numpy as np
import bayesflow as bf
import keras

SEED = 2026
N_TRIALS = 1000
MAX_T = 3.0
DT = 0.02

PRIORS = {
    "nu":    {"shape": 5, "scale": 0.5},
    "alpha1":{"shape": 5, "scale": 0.2},
    "alpha2_gap": {"time_scale": 6.0, "time_shape": 5},
    "tau":   {"scale": 0.15},
}

rng = np.random.default_rng(SEED)


INFO:jax._src.xla_bridge:Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: libtpu.so: cannot open shared object file: No such file or directory
INFO:bayesflow:Using backend 'jax'
/usr/local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Priors

In [2]:
def prior():
    nu = rng.gamma(shape=PRIORS["nu"]["shape"], scale=PRIORS["nu"]["scale"], size=2)
    alpha1 = rng.gamma(shape=PRIORS["alpha1"]["shape"], scale=PRIORS["alpha1"]["scale"])

    time_scale = PRIORS["alpha2_gap"]["time_scale"]
    time_shape = PRIORS["alpha2_gap"]["time_shape"]
    extra_time = rng.gamma(shape=time_shape, scale=time_scale / time_shape)
    gap = extra_time * np.mean(nu)
    alpha2 = alpha1 + gap

    tau = rng.exponential(scale=PRIORS["tau"]["scale"])
    return {"nu": nu, "alpha1": alpha1, "alpha2": alpha2, "tau": tau}

print(prior())


{'nu': array([1.57722685, 0.8259513 ]), 'alpha1': 1.2371612415922888, 'alpha2': np.float64(7.040398264159797), 'tau': 0.1367541617600175}


## 2. Simulator

In [3]:
def context():
    return {"n": N_TRIALS}

def race_to_threshold(drift, threshold, n, max_t=MAX_T, dt=DT):
    n_steps = int(max_t / dt)
    sqrt_dt = np.sqrt(dt)
    drift_arr = np.full(n, drift) if np.isscalar(drift) else drift
    evidence = np.zeros(n)
    crossed = np.zeros(n, dtype=bool)
    crossing_time = np.full(n, max_t)

    for step in range(1, n_steps + 1):
        t = step * dt
        noise = rng.normal(scale=sqrt_dt, size=n)
        evidence += drift_arr * dt + noise
        newly_crossed = (~crossed) & (evidence >= threshold)
        crossing_time[newly_crossed] = t
        crossed |= newly_crossed
        if crossed.all():
            break
    return crossing_time

def simulate_dataset(nu, alpha1, alpha2, tau, n=N_TRIALS):
    t0 = race_to_threshold(nu[0], alpha1, n)
    t1 = race_to_threshold(nu[1], alpha1, n)
    correct = np.where(t0 < t1, 0.0, 1.0)
    first_response_time = np.minimum(t0, t1) + tau
    loser_drift = np.where(t0 < t1, nu[1], nu[0])
    t2 = race_to_threshold(loser_drift, alpha2, n)
    double_flag = (t2 < MAX_T).astype(float)
    second_response_time = np.where(double_flag == 1.0, t2 + tau, 0.0)
    return {
        "first_response_time": first_response_time,
        "correct": correct,
        "double_flag": double_flag,
        "second_response_time": second_response_time,
    }

p = prior()
data = simulate_dataset(p["nu"], p["alpha1"], p["alpha2"], p["tau"])
print("double response rate:", data["double_flag"].mean())


double response rate: 0.027


## 3. BayesFlow Simulator + Adapter

In [4]:
simulator = bf.make_simulator([context, prior, simulate_dataset])

adapter = (
    bf.Adapter()
    .as_set(["first_response_time", "correct", "double_flag", "second_response_time"])
    .constrain(["nu", "alpha1", "alpha2", "tau"], lower=0)
    .standardize(include="nu", mean=2.502, std=1.105)
    .standardize(include="alpha1", mean=1.005, std=0.449)
    .standardize(include="alpha2", mean=16.087, std=8.544)
    .standardize(include="tau", mean=0.159, std=0.165)
    .concatenate(["nu", "alpha1", "alpha2", "tau"], into="inference_variables")
    .concatenate(["first_response_time", "correct", "double_flag", "second_response_time"], into="summary_variables")
)

batch = simulator.sample(5)
adapted = adapter(batch)
for k, v in adapted.items():
    print(k, v.shape)


n (5, 1)
inference_variables (5, 5)
summary_variables (5, 1000, 4)


## 4. Workflow + Training
Training from scratch here takes ~2 hours. To skip straight to using an already-trained model, go to section 5 instead.

In [ ]:
workflow = bf.BasicWorkflow(
    simulator=simulator,
    adapter=adapter,
    inference_network=bf.networks.CouplingFlow(),
    summary_network=bf.networks.DeepSet(),
    inference_variables=["nu", "alpha1", "alpha2", "tau"],
    inference_conditions=["n"],
    summary_variables=["first_response_time", "correct", "double_flag", "second_response_time"],
)

history = workflow.fit_online(epochs=320, num_batches_per_epoch=50, batch_size=32)
workflow.approximator.save("../../results/trained_model.keras")


## 5. Load Already-Trained Model (fast path)

In [5]:
approximator = keras.saving.load_model("../../results/trained_model.keras")
print("model loaded")


model loaded


## 6. Parameter Recovery Check

In [6]:
true_params = prior()
trial_data = simulate_dataset(true_params["nu"], true_params["alpha1"], true_params["alpha2"], true_params["tau"])

fake_data = {
    "n": np.array([[1000]]),
    "nu": true_params["nu"].reshape(1, -1),
    "alpha1": np.array([[true_params["alpha1"]]]),
    "alpha2": np.array([[true_params["alpha2"]]]),
    "tau": np.array([[true_params["tau"]]]),
    "first_response_time": trial_data["first_response_time"].reshape(1, -1),
    "correct": trial_data["correct"].reshape(1, -1),
    "double_flag": trial_data["double_flag"].reshape(1, -1),
    "second_response_time": trial_data["second_response_time"].reshape(1, -1),
}

posterior = approximator.sample(conditions=fake_data, num_samples=500)
print("true:", true_params)
print("estimated:", {k: posterior[k].mean(axis=1) for k in ["nu", "alpha1", "alpha2", "tau"]})


Sampling: 100%|██████████| 1/1 [00:01<00:00,  1.27s/batch]

true: {'nu': array([2.34614326, 1.49116563]), 'alpha1': 1.465712118868907, 'alpha2': np.float64(10.96903260563568), 'tau': 0.19684272828296245}
estimated: {'nu': array([[2.28609272, 1.43855342]]), 'alpha1': array([[1.30093389]]), 'alpha2': array([[11.01658116]]), 'tau': array([[0.23251468]])}


## 7. Diagnostics (calibration, SBC) - TODO, diagnostics pair

In [ ]:
# TODO: calibration plots, SBC, posterior contraction using approximator.sample()


## 8. Real Data Fitting (Evans et al. 2020) - TODO, diagnostics pair

In [ ]:
# TODO: load real dataset, run approximator.sample() on it, compare to Evans et al. (2020)
